# 02 — Price Integrity & Cleaning

**Project:** AI-Based NIFTY 50 Portfolio Risk Prediction & Early Warning System

### Objective
Build a conservative, auditable price-quality layer before calculating portfolio returns and risk features.

### Important principle
We preserve the original CSV as the raw source. We do **not** overwrite it.


## 1. Why this step matters

A stock split or a bad historical price can look like a huge gain/loss if we calculate returns naively.

For example:

`₹2 → ₹250 → ₹2`

would look like a massive rally followed by a crash, even though it may simply be a bad historical observation.

Our goal is therefore:

**Raw data → Quality checks → Flags → Conservative clean dataset**


In [ ]:
import pandas as pd
import numpy as np

RAW_PATH = "../data/raw/nifty50_historical_data.csv"

df = pd.read_csv(RAW_PATH)
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
df = df.sort_values(["Ticker", "Date"]).reset_index(drop=True)

print("Shape:", df.shape)
print("Original CSV columns:", 25)
print("Tickers:", df["Ticker"].nunique())
print("Date range:", df["Date"].min(), "to", df["Date"].max())


## 2. Recalculate returns ourselves

The dataset already contains `Daily_Return`, but we do not want our pipeline to depend on a precomputed feature.

We calculate:

\[
R_t = \frac{Close_t}{Close_{t-1}} - 1
\]

and later use this canonical return series for feature engineering.


In [ ]:
df["Calculated_Return"] = df.groupby("Ticker")["Close"].pct_change()

# Verify the supplied Daily_Return, if present
mask = df["Daily_Return"].notna() & df["Calculated_Return"].notna()
max_abs_difference = (
    df.loc[mask, "Daily_Return"] - df.loc[mask, "Calculated_Return"]
).abs().max()

print("Maximum difference vs supplied Daily_Return:", max_abs_difference)


## 3. Validate OHLC prices

Basic rules:

- Low ≤ Open
- Low ≤ Close
- High ≥ Open
- High ≥ Close
- Low ≤ High
- Core OHLC values should not be missing
- Core prices should be positive


In [ ]:
df["Flag_Missing_Core_Price"] = df[["Open", "High", "Low", "Close"]].isna().any(axis=1)
df["Flag_NonPositive_Price"] = (df[["Open", "High", "Low", "Close"]] <= 0).any(axis=1)

df["Flag_OHLC_Invalid"] = (
    (df["Low"] > df["Open"]) |
    (df["Low"] > df["Close"]) |
    (df["High"] < df["Open"]) |
    (df["High"] < df["Close"]) |
    (df["Low"] > df["High"])
)

print("Missing core price rows:", df["Flag_Missing_Core_Price"].sum())
print("Non-positive price rows:", df["Flag_NonPositive_Price"].sum())
print("Invalid OHLC rows:", df["Flag_OHLC_Invalid"].sum())


## 4. Detect suspicious price spikes

We use a **conservative** rule for automatic cleaning.

A row is considered a confirmed spike-reversal anomaly when:

1. Absolute return > 30%
2. The next available close returns to within 5% of the pre-spike close
3. No stock split is recorded on that row

This is deliberately stricter than simply deleting every return above 30%, because genuine market events can be large.

`Extreme return` is therefore a **review flag**, while `Spike reversal` is an automatic cleaning flag.


In [ ]:
prev_close = df.groupby("Ticker")["Close"].shift(1)
next_close = df.groupby("Ticker")["Close"].shift(-1)

next_valid_return_to_prev = (next_close / prev_close) - 1

df["Flag_Extreme_Return"] = df["Calculated_Return"].abs().gt(0.30)

df["Flag_Spike_Reversal"] = (
    df["Calculated_Return"].abs().gt(0.30) &
    prev_close.notna() &
    next_close.notna() &
    next_valid_return_to_prev.abs().lt(0.05) &
    df["Stock_Split"].isna()
)

print("Extreme-return rows:", df["Flag_Extreme_Return"].sum())
print("Confirmed spike-reversal rows:", df["Flag_Spike_Reversal"].sum())


## 5. Create quality flags

We keep two ideas separate:

- **Flagged:** clearly problematic observations that should not enter the clean return series.
- **Extreme but not confirmed:** unusual observations that remain in the audit file for manual review.

This avoids blindly deleting legitimate market events.


In [ ]:
flag_cols = [
    "Flag_Missing_Core_Price",
    "Flag_NonPositive_Price",
    "Flag_OHLC_Invalid",
    "Flag_Spike_Reversal",
]

df["Data_Quality_Flag"] = df[flag_cols].any(axis=1)

def quality_reason(row):
    reasons = []
    if row["Flag_Missing_Core_Price"]:
        reasons.append("missing_core_price")
    if row["Flag_NonPositive_Price"]:
        reasons.append("nonpositive_price")
    if row["Flag_OHLC_Invalid"]:
        reasons.append("invalid_ohlc")
    if row["Flag_Spike_Reversal"]:
        reasons.append("spike_reversal")
    return ";".join(reasons) if reasons else "ok"

df["Data_Quality_Reason"] = df.apply(quality_reason, axis=1)

print(df["Data_Quality_Reason"].value_counts())


## 6. Build the clean price dataset

We exclude only observations that fail the quality rules.

We **do not** use current fundamental fields here.

We also recalculate the canonical return after cleaning.


In [ ]:
clean = df.loc[~df["Data_Quality_Flag"]].copy()

clean["Return"] = clean.groupby("Ticker")["Close"].pct_change()

keep_cols = [
    "Date", "Ticker", "Company_Name", "Sector",
    "Open", "High", "Low", "Close", "Volume",
    "Dividend", "Stock_Split", "Return",
    "Flag_Extreme_Return",
    "Data_Quality_Flag", "Data_Quality_Reason"
]

clean = clean[keep_cols]

print("Clean dataset shape:", clean.shape)


## 7. Save outputs

This notebook produces:

- `clean_price_data.csv` — the conservative clean price dataset
- `price_data_with_quality_flags.csv` — raw observations plus quality flags
- `price_integrity_audit_summary.csv` — audit summary
- `flagged_price_observations.csv` — unusual observations for review

The original raw CSV is never modified.


In [ ]:
clean.to_csv("../data/processed/clean_price_data.csv", index=False)
df.to_csv("../data/processed/price_data_with_quality_flags.csv", index=False)

summary = pd.DataFrame({
    "Metric": [
        "Raw rows",
        "Raw columns",
        "Unique tickers",
        "Unique sectors",
        "Minimum date",
        "Maximum date",
        "Duplicate rows",
        "Duplicate Date+Ticker",
        "Missing core OHLC rows",
        "Non-positive price rows",
        "Invalid OHLC rows",
        "Extreme return rows (>30%)",
        "Confirmed spike-reversal rows",
        "Rows removed",
        "Clean rows",
    ],
    "Value": [
        len(df),
        df.shape[1],
        df["Ticker"].nunique(),
        df["Sector"].nunique(),
        df["Date"].min(),
        df["Date"].max(),
        df.duplicated().sum(),
        df.duplicated(["Date", "Ticker"]).sum(),
        df["Flag_Missing_Core_Price"].sum(),
        df["Flag_NonPositive_Price"].sum(),
        df["Flag_OHLC_Invalid"].sum(),
        df["Flag_Extreme_Return"].sum(),
        df["Flag_Spike_Reversal"].sum(),
        df["Data_Quality_Flag"].sum(),
        len(clean),
    ]
})

summary.to_csv("../reports/price_integrity_audit_summary.csv", index=False)

flagged = df.loc[
    df["Data_Quality_Flag"] | df["Flag_Extreme_Return"],
    [
        "Date", "Ticker", "Company_Name", "Close",
        "Calculated_Return", "Stock_Split", "Dividend",
        "Flag_Extreme_Return", "Data_Quality_Flag",
        "Data_Quality_Reason"
    ]
].sort_values(["Date", "Ticker"])

flagged.to_csv("../reports/flagged_price_observations.csv", index=False)

summary


## 8. Important modeling decision

For later feature engineering:

### We will use
- `Close`
- `Open`
- `High`
- `Low`
- `Volume`
- `Return`

### We will not use the dataset's precomputed
- `Daily_Return`
- `Volatility_20D`
- `MA_50`
- `MA_200`

We will calculate our own features from the clean price data.

### We will also keep
`Flag_Extreme_Return`

as an audit/review field rather than automatically deleting every extreme move.
